# Proposal: quantum encodings on Track A

## Goal

Extend the Track B quantum feature tracks (`Q_{encoding}_{getter}`, built in `quantum_data_prep.ipynb`) to the full stationary panel (Track A, 123 features × 767 months). The question is whether quantum encodings add value when they get the full panel instead of a few hand-picked indicators.

## Current setup (Track B)

- Inputs: `track_b_{target}.parquet` (10–12 features), causal (expanding) min-max scaled to [0, 1], then mapped to angles in [0, 2π].
- Encodings: `su` (EfficientSU2, `ceil(n/2)` qubits), `heisenberg` (Trotterized XX+YY+ZZ chain, `n+1` qubits, 6 steps), `zz` (ZZ-only chain, `n+1` qubits).
- Getters: `pce` (9 correlators), `pauli` (⟨X⟩, ⟨Y⟩, ⟨Z⟩ per qubit), `single_bit`, `bitstring` (skipped above 8 qubits).
- Simulation: Aer statevector, estimator precision 0.01, 4096 shots, no noise. All 48 combinations took 17.8 min (max 13 qubits, slowest 167 s per track).

## Track B results

_To be completed._

## Scaling problem

| Encoding | Track B qubits | Track A qubits |
|---|---|---|
| su | 5–6 | ~62 |
| heisenberg / zz | 11–13 | ~124 |

Statevector memory doubles with each qubit and tops out near 30 qubits on a laptop, so the Track B pipeline cannot run on Track A unchanged.

## Proposed approach

1. Full panel with Aer `matrix_product_state` (MPS). Its cost depends on entanglement, not qubit count.
   - `zz`: each bond carries one diagonal ZZ rotation, so bond dimension ≤ 2. Expected runtime: minutes.
   - `su`: switch `entanglement="full"` to `"reverse_linear"` (same unitary, n−1 nearest-neighbour CNOTs), giving bond dimension ≤ 2. Expected runtime: minutes.
   - `heisenberg`: exact bond dimension can reach 4^6 = 4096, which is infeasible. Run it truncated (e.g. `max_bond_dimension=64`) and report it as approximate.
   - `bitstring`: dropped (2^62+ columns).
2. Reduced panel (fallback or comparison). Encode the first 15 principal components (as in Track D mini): ~8 qubits for su and 16 for heisenberg/zz, both feasible as statevector. PCA must be fit causally (expanding window) so precomputed features have no look-ahead.

## Evaluation

- Same evaluation loop, folds, targets (INDPRO, PAYEMS, CPIAUCSL, S&P 500) and horizons (1, 3, 6, 12) as the existing tracks.
- Compare against (a) classical models on Track A, (b) the same encodings on Track B, and (c) the PCA variant, using MASE / relative RMSE and the existing rank plots.
- Validate MPS output against exact statevector on a feature subset small enough for both (≤ 25 qubits).

## Compute estimate

Not yet benchmarked. Planned: time 25 rows per encoding with MPS before committing to the full run. Expected: minutes for zz and su, hours or more for truncated heisenberg depending on bond dimension.

## Risks

- Truncated heisenberg features may not reflect the intended encoding.
- 62–124 rotation angles for 767 rows could give near-orthogonal states and uninformative features (concentration), especially for `pce` and `single_bit`.
- Results without noise do not show performance on hardware at this width.